# Shot-Count Convergence Analysis — N_SHOTS = 100,000

Runs both circuit topologies at 100k shots, computes Wilson score CIs,
saves histogram PDFs, and prints the top-5 table + Spearman ρ for Supplementary Table S1.

**Outputs** (written to `figures/`):
- `100000_shot_histogram_co.pdf` → Supplementary Fig. S1
- `100000_shot_histogram_mo.pdf` → Supplementary Fig. S2
- Console output → fill Supplementary Table S1

In [8]:
import numpy as np
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — safe for saving PDFs
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import os, pathlib

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qsim_cells.generative import (
    create_rotation_circuit,
    concatenate_circuits_with_separate_measurements,
    add_crx_and_measurements_to_circuit,
)

# ── Parameters (must match main notebook) ────────────────────────────────────
MY_SEED  = 42
N_SHOTS  = 100_000
N_SHOTS_2K = 2_000   # reference run for convergence comparison

ANG_CT1 = np.array([0.2, 0.1, 0.4, 0.9, 0.8]) * np.pi
ANG_CT2 = np.array([0.2, 0.3, 0.2, 0.7, 0.5]) * np.pi

INTERACTION_CASE1 = [(3, 5), (5, 7), (7, 0)]   # L1 — inter-state cascade
INTERACTION_CASE2 = [(2, 1)]                     # L2 — non-interacting control

FIG_DIR = pathlib.Path('figures')
FIG_DIR.mkdir(exist_ok=True)

np.random.seed(MY_SEED)
print(f'N_SHOTS={N_SHOTS}  seed={MY_SEED}  fig_dir={FIG_DIR.resolve()}')

N_SHOTS=100000  seed=42  fig_dir=C:\Users\selim\Escritorio\qSimCells_review\qsimcells_review_cowork\code\qSimCells\figures


In [9]:
# ── Helper: run circuit and return raw counts dict ───────────────────────────

def run_circuit(ang_ct1, ang_ct2, interaction_map, n_shots, seed, backend=None):
    """Returns (counts_ct1, counts_ct2) as dicts {bitstring: count}."""
    np.random.seed(seed)
        
    circ1    = create_rotation_circuit(ang_ct1)
    circ2    = create_rotation_circuit(ang_ct2)
    combined = concatenate_circuits_with_separate_measurements(circ1, circ2)
    final    = add_crx_and_measurements_to_circuit(combined, circ1.num_qubits, interaction_map)

    if backend is None:
        backend = AerSimulator(seed_simulator=seed)

    try:
        pm      = generate_preset_pass_manager(backend=backend, optimization_level=3)
        qc_comp = pm.run(final)
    except Exception:
        qc_comp = final
    
    result = Sampler(mode=backend).run([qc_comp], shots=n_shots).result()[0]

    reg_names       = [cr.name for cr in final.cregs]
    counts_ct1 = result.data.c_measure1.get_counts() if 'c_measure1' in reg_names else None
    counts_ct2 = result.data.c_measure2.get_counts() if 'c_measure2' in reg_names else None

    return counts_ct1, counts_ct2


def counts_to_freq(counts_dict, n_qubits):
    """Convert counts dict → numpy array of length 2**n_qubits (indexed by decimal state)."""
    n_states = 2 ** n_qubits
    total    = sum(counts_dict.values())
    freq     = np.zeros(n_states)
    for bitstr, cnt in counts_dict.items():
        idx = int(bitstr[::-1], 2)   # reverse for little-endian
        freq[idx] = cnt / total
    return freq


def wilson_ci(p, n, z=1.96):
    """Wilson score CI half-width for a proportion p estimated from n trials."""
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2*n)) / denom
    half   = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return centre, half

print('Helpers defined.')

Helpers defined.


In [10]:
# ── Run both topologies at 2k and 100k ───────────────────────────────────────
print('Running L1 (inter-state cascade) at 2k shots...')
c1_2k_ct1, c1_2k_ct2 = run_circuit(ANG_CT1, ANG_CT2, INTERACTION_CASE1, N_SHOTS_2K, MY_SEED)

print('Running L1 at 100k shots...')
c1_100k_ct1, c1_100k_ct2 = run_circuit(ANG_CT1, ANG_CT2, INTERACTION_CASE1, N_SHOTS, MY_SEED)

print('Running L2 (non-interacting control) at 100k shots...')
c2_100k_ct1, c2_100k_ct2 = run_circuit(ANG_CT1, ANG_CT2, INTERACTION_CASE2, N_SHOTS, MY_SEED)

print('All circuits done.')

Running L1 (inter-state cascade) at 2k shots...
Running L1 at 100k shots...
Running L2 (non-interacting control) at 100k shots...
All circuits done.


In [11]:
# ── Plot histogram with Wilson CI error bars and save as PDF ─────────────────

def plot_histogram_with_ci(counts_ct1, counts_ct2, n_shots, title, save_path):
    """Single-run histogram: 2 subplots (c_measure1 left, c_measure2 right)."""
    n_qubits = 5
    n_states = 2 ** n_qubits
    states   = np.arange(n_states)

    # Guard: missing register → empty distribution
    if counts_ct1 is None: counts_ct1 = {}
    if counts_ct2 is None: counts_ct2 = {}

    freq1 = counts_to_freq(counts_ct1, n_qubits)
    freq2 = counts_to_freq(counts_ct2, n_qubits)

    def _ci_arr(freq, n):
        _, hw = zip(*[wilson_ci(p, n) for p in freq])
        return np.array(hw)

    ci1 = _ci_arr(freq1, n_shots)
    ci2 = _ci_arr(freq2, n_shots)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, freq, ci, reg_label, color in zip(
            axes, [freq1, freq2], [ci1, ci2],
            ['c_measure1 / CT1 (5 qubits)', 'c_measure2 / CT2 (5 qubits)'],
            ['#2E86AB', '#E84855']):
        ax.bar(states, freq, color=color, alpha=0.75)
        ax.errorbar(states, freq, yerr=ci, fmt='none',
                    ecolor='black', elinewidth=0.8, capsize=2)
        ax.set_xlabel('Basis state (decimal index)', fontsize=13)
        ax.set_ylabel('Probability', fontsize=13)
        ax.set_title(reg_label, fontsize=14)
        ax.set_xticks(states[::2])
        ax.tick_params(axis='x', labelsize=10)

    fig.suptitle(f'{title}  ($N_{{\\mathrm{{shots}}}}={n_shots:,}$, seed={MY_SEED})\n'
                 f'Error bars: 95% Wilson CI', fontsize=14)
    plt.tight_layout()
    fig.savefig(save_path, format='pdf', bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f'Saved: {save_path}')


def plot_combined_convergence(counts_2k_ct1, counts_2k_ct2,
                               counts_100k_ct1, counts_100k_ct2,
                               topology_label, save_path):
    """
    2×2 grid PDF: rows = shot count (2k top, 100k bottom),
    cols = register (c_measure1 left, c_measure2 right).
    Error bars shown only on 100k row (Wilson 95% CI).
    """
    n_qubits = 5
    n_states = 2 ** n_qubits
    states   = np.arange(n_states)

    for attr in ['counts_2k_ct1','counts_2k_ct2','counts_100k_ct1','counts_100k_ct2']:
        if locals()[attr] is None:
            locals()[attr] = {}

    f2k_1  = counts_to_freq(counts_2k_ct1  or {}, n_qubits)
    f2k_2  = counts_to_freq(counts_2k_ct2  or {}, n_qubits)
    f100_1 = counts_to_freq(counts_100k_ct1 or {}, n_qubits)
    f100_2 = counts_to_freq(counts_100k_ct2 or {}, n_qubits)

    def _ci(freq, n):
        _, hw = zip(*[wilson_ci(p, n) for p in freq])
        return np.array(hw)

    ci_1    = _ci(f100_1, N_SHOTS)
    ci_2    = _ci(f100_2, N_SHOTS)
    ci_2k_1 = _ci(f2k_1,  N_SHOTS_2K)
    ci_2k_2 = _ci(f2k_2,  N_SHOTS_2K)

    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex='col')

    panel_cfg = [
        (axes[0,0], f2k_1,  ci_2k_1, N_SHOTS_2K, '#2E86AB', 'c_measure1 / CT1'),
        (axes[0,1], f2k_2,  ci_2k_2, N_SHOTS_2K, '#E84855', 'c_measure2 / CT2'),
        (axes[1,0], f100_1, ci_1,  N_SHOTS,    '#2E86AB', 'c_measure1 / CT1'),
        (axes[1,1], f100_2, ci_2,  N_SHOTS,    '#E84855', 'c_measure2 / CT2'),
    ]
    for ax, freq, ci, n, color, reg_label in panel_cfg:
        ax.bar(states, freq, color=color, alpha=0.75)
        if ci is not None:
            ax.errorbar(states, freq, yerr=ci, fmt='none',
                        ecolor='black', elinewidth=0.8, capsize=2)
        ax.set_title(reg_label, fontsize=13)
        ax.set_xticks(states[::4])
        ax.tick_params(axis='x', labelsize=10)
        ax.set_ylabel('Probability', fontsize=12)

    for ax in axes[1]:
        ax.set_xlabel('Basis state (decimal index)', fontsize=12)

    row_labels = [f'$N = {N_SHOTS_2K:,}$ shots',
                  f'$N = {N_SHOTS:,}$ shots  (95\\% Wilson CI)']
    for row, label in zip([0, 1], row_labels):
        axes[row, 0].set_ylabel(f'{label}\nProbability', fontsize=11)

    fig.suptitle(f'{topology_label}\n'
                 f'Convergence: $N_{{2k}}={N_SHOTS_2K:,}$ vs $N_{{100k}}={N_SHOTS:,}$ shots  '
                 f'(seed = {MY_SEED})',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    fig.savefig(save_path, format='pdf', bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f'Saved: {save_path}')


# ── individual 100k plots ─────────────────────────────────────────────────
plot_histogram_with_ci(
    c1_100k_ct1, c1_100k_ct2, N_SHOTS,
    title='Co-culture — inter-state cascade $L_1$',
    save_path=FIG_DIR / '100000_shot_histogram_co.pdf'
)
plot_histogram_with_ci(
    c2_100k_ct1, c2_100k_ct2, N_SHOTS,
    title='Mono-culture — non-interacting control $L_2$',
    save_path=FIG_DIR / '100000_shot_histogram_mo.pdf'
)

# ── combined convergence figure (2k vs 100k, both registers) ─────────────
plot_combined_convergence(
    c1_2k_ct1, c1_2k_ct2,
    c1_100k_ct1, c1_100k_ct2,
    topology_label='Co-culture — inter-state cascade $L_1$',
    save_path=FIG_DIR / 'qsimcells_figs1.pdf'
)


Saved: figures\100000_shot_histogram_co.pdf
Saved: figures\100000_shot_histogram_mo.pdf
Saved: figures\qsimcells_figs1.pdf


In [12]:
# ── Convergence table: top-5 states + Spearman ρ — CT1 and CT2 ─────────────
# Uses L1 (co-culture) both registers — 5 qubits each, 32 states
import csv

n_qubits = 5
N2K  = N_SHOTS_2K
N100 = N_SHOTS

# ── CT1 ──────────────────────────────────────────────────────────────────────
freq_2k_1  = counts_to_freq(c1_2k_ct1,   n_qubits)
freq_100_1 = counts_to_freq(c1_100k_ct1, n_qubits)

counts_2k_arr1 = np.zeros(2**n_qubits)
for bitstr, cnt in c1_2k_ct1.items():
    counts_2k_arr1[int(bitstr[::-1], 2)] = cnt

_, ci_100_1 = zip(*[wilson_ci(p, N100) for p in freq_100_1])
ci_100_1 = np.array(ci_100_1)
top5_1 = np.argsort(freq_100_1)[::-1][:5]

rho1, pval1 = spearmanr(freq_2k_1, freq_100_1)

# ── CT2 ──────────────────────────────────────────────────────────────────────
freq_2k_2  = counts_to_freq(c1_2k_ct2,   n_qubits)
freq_100_2 = counts_to_freq(c1_100k_ct2, n_qubits)

counts_2k_arr2 = np.zeros(2**n_qubits)
for bitstr, cnt in c1_2k_ct2.items():
    counts_2k_arr2[int(bitstr[::-1], 2)] = cnt

_, ci_100_2 = zip(*[wilson_ci(p, N100) for p in freq_100_2])
ci_100_2 = np.array(ci_100_2)
top5_2 = np.argsort(freq_100_2)[::-1][:5]

rho2, pval2 = spearmanr(freq_2k_2, freq_100_2)

# ── Print combined table ─────────────────────────────────────────────────────
SEP = '─' * 80
HDR = (f"{'State':>6}  {'Count(2k)':>10}  {'Freq(2k,%)':>11}  "
       f"{'Count(100k)':>12}  {'Freq(100k,%)':>13}  {'95%CI(±%)':>11}")

for label, top5, arr2k, f2k, f100, ci100, rho, pval in [
    ('CT1 — c_measure1', top5_1, counts_2k_arr1, freq_2k_1, freq_100_1, ci_100_1, rho1, pval1),
    ('CT2 — c_measure2', top5_2, counts_2k_arr2, freq_2k_2, freq_100_2, ci_100_2, rho2, pval2),
]:
    print(f'\n{label}')
    print(SEP)
    print(HDR)
    print(SEP)
    for idx in top5:
        print(f'{idx:>6}  {int(arr2k[idx]):>10}  '
              f'{f2k[idx]*100:>10.2f}%  '
              f'{int(round(f100[idx]*N100)):>12,}  '
              f'{f100[idx]*100:>12.2f}%  '
              f'{ci100[idx]*100:>10.3f}%')
    print(SEP)
    print(f'Spearman rho = {rho:.4f}  (p = {pval:.2e})')

# ── Export combined CSV ──────────────────────────────────────────────────────
ci_csv_path = FIG_DIR / 'shot_convergence_ci.csv'
with open(ci_csv_path, 'w', newline='') as _f:
    _w = csv.writer(_f)
    _w.writerow(['Register','State','Count_2k','Freq_2k_pct','Count_100k','Freq_100k_pct','CI_95_pct'])
    for reg, top5, arr2k, f2k, f100, ci100 in [
        ('CT1', top5_1, counts_2k_arr1, freq_2k_1, freq_100_1, ci_100_1),
        ('CT2', top5_2, counts_2k_arr2, freq_2k_2, freq_100_2, ci_100_2),
    ]:
        for _idx in top5:
            _w.writerow([
                reg, int(_idx),
                int(arr2k[_idx]),
                round(f2k[_idx]*100, 2),
                int(round(f100[_idx]*N100)),
                round(f100[_idx]*100, 2),
                round(ci100[_idx]*100, 3),
            ])
print(f'\nCSV saved -> {ci_csv_path}')



CT1 — c_measure1
────────────────────────────────────────────────────────────────────────────────
 State   Count(2k)   Freq(2k,%)   Count(100k)   Freq(100k,%)    95%CI(±%)
────────────────────────────────────────────────────────────────────────────────
    19         844       42.20%        42,878         42.88%       0.307%
    23         456       22.80%        22,719         22.72%       0.260%
     3         263       13.15%        13,204         13.20%       0.210%
     7         154        7.70%         7,062          7.06%       0.159%
    18          92        4.60%         4,630          4.63%       0.130%
────────────────────────────────────────────────────────────────────────────────
Spearman rho = 0.9324  (p = 8.47e-15)

CT2 — c_measure2
────────────────────────────────────────────────────────────────────────────────
 State   Count(2k)   Freq(2k,%)   Count(100k)   Freq(100k,%)    95%CI(±%)
────────────────────────────────────────────────────────────────────────────────
   